# CodeAgent — Qwen2.5-Coder-7B QLoRA Training on Kaggle

Train **`Qwen/Qwen2.5-Coder-7B-Instruct`** on your prepared dataset with **Unsloth 4-bit QLoRA** using Kaggle's free GPU.

---
### ⚙️ Required Kaggle Settings (Right Panel -> Session options):
1. **Accelerator**: Select **GPU T4 x 2** (or GPU P100)
2. **Internet**: Toggle to **Internet on** (required to download model & packages)
3. **Environment**: Standard (Python 3.10)


## 1. Verify GPU Hardware
Confirms NVIDIA GPU (T4 / P100) is active.

In [ ]:
!nvidia-smi
import torch
print("\nCUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        prop = torch.cuda.get_device_properties(i)
        print(f"  Device {i}: {prop.name} ({prop.total_memory / (1024**3):.1f} GB VRAM)")

## 2. Install Fast QLoRA Dependencies (BitsAndBytes & Friends)

In [ ]:
%%capture
!pip install -U "bitsandbytes>=0.46.1"
!pip install -U transformers accelerate peft trl datasets pyyaml
!pip install --no-deps "unsloth @ git+https://github.com/unslothai/unsloth.git"
print("✓ Dependencies installed successfully!")

## 3. Unpack Dataset & Training Code
Locates `training` and `dataset` from your uploaded Kaggle dataset and copies them into `/kaggle/working/codeagent`.

In [ ]:
import os, zipfile, shutil
from pathlib import Path

ROOT = Path("/kaggle/working/codeagent")
ROOT.mkdir(parents=True, exist_ok=True)

# 1. Check if uploaded as a raw .zip archive
for p in Path("/kaggle").rglob("*.zip"):
    if "bundle" in p.name.lower() or "codeagent" in p.name.lower():
        print(f"Extracting zip archive: {p}")
        with zipfile.ZipFile(p, "r") as zf:
            zf.extractall(ROOT)
        break

# 2. Locate data.py in /kaggle/input (Kaggle auto-extracts uploaded datasets)
data_files = list(Path("/kaggle/input").rglob("data.py"))
if data_files:
    training_src = data_files[0].parent
    project_src = training_src.parent
    print(f"Found project root at: {project_src}")
    
    if (project_src / "training").exists():
        shutil.copytree(project_src / "training", ROOT / "training", dirs_exist_ok=True)
    if (project_src / "dataset").exists():
        shutil.copytree(project_src / "dataset", ROOT / "dataset", dirs_exist_ok=True)
    print(f"✓ Copied 'training' and 'dataset' into {ROOT}")
elif not (ROOT / "training" / "data.py").exists():
    print("⚠️ Could not find data.py in /kaggle/input!")
    print("Items in /kaggle/input:", list(Path("/kaggle/input").iterdir()) if Path("/kaggle/input").exists() else [])

# Set working directory for both Python and Notebook shell
os.chdir(ROOT)
%cd /kaggle/working/codeagent
print(f"Current working directory: {os.getcwd()}")
print("Directory contents:", os.listdir(ROOT))
if (ROOT / "training").exists():
    print("Training folder contents:", os.listdir(ROOT / "training"))

## 4. Verify Dataset & Approval Gates

In [ ]:
import sys
from pathlib import Path

ROOT = Path("/kaggle/working/codeagent")
training_path = str(ROOT / "training")
if training_path not in sys.path:
    sys.path.insert(0, training_path)

from data import check_pilot_approvals, load_pilot_splits

check_pilot_approvals(str(ROOT / "dataset/source"), "Qwen/Qwen2.5-Coder-7B-Instruct", "c03e6d358207e414f1eca0bb1891e29f1db0e242")
splits = load_pilot_splits(str(ROOT / "dataset/source"))
print("✓ Dataset validation & approvals PASSED!")
for name, rows in splits.items():
    print(f"  - {name}: {len(rows)} samples, {len(set(r['group_id'] for r in rows))} independent groups")

## 5. Run Training (7B Model, QLoRA 4-bit)

- `smoke`: Fast 2-3 step dry-run to test loading, memory & loss masking (~2 min)
- `pilot`: Full training run with cosine scheduler and assistant-only loss masking

In [ ]:
# Choose 'smoke' (2-step test) or 'pilot' (full run)
MODE = "pilot"  # ["smoke", "pilot"]

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True CUDA_VISIBLE_DEVICES=0 python {ROOT}/training/train.py --mode {MODE} --root {ROOT}

## 6. Test Fine-Tuned Model Inference
Generates code with the newly trained LoRA adapter!

In [ ]:
import json
from pathlib import Path
from transformers import AutoTokenizer

latest_file = ROOT / f"training/runs/latest-{MODE}.json"
if latest_file.exists():
    run_id = json.loads(latest_file.read_text())["run_id"]
    model_path = ROOT / f"training/models/{run_id}"
    print(f"Testing adapter from: {model_path}")
    
    try:
        from unsloth import FastLanguageModel
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=str(model_path),
            max_seq_length=4096,
            dtype=None,
            load_in_4bit=True,
        )
        FastLanguageModel.for_inference(model)
    except Exception:
        from transformers import AutoModelForCausalLM
        from peft import PeftModel
        tokenizer = AutoTokenizer.from_pretrained(str(model_path))
        base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct", load_in_4bit=True, device_map="auto")
        model = PeftModel.from_pretrained(base, str(model_path))

    messages = [
        {"role": "system", "content": "You are CodeAgent, an expert autonomous software engineer and tool user."},
        {"role": "user", "content": "Write an optimized Python function find_peak_element(nums) that finds a peak element in O(log n) time using binary search. Include docstring and doctests."}
    ]
    encoded = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
    if isinstance(encoded, dict) or hasattr(encoded, "items"):
        input_ids = encoded["input_ids"].to("cuda")
    else:
        input_ids = encoded.to("cuda")
        
    outputs = model.generate(input_ids=input_ids, max_new_tokens=512, use_cache=True)
    resp = tokenizer.batch_decode(outputs)[0]
    clean_resp = resp.split("<|im_start|>assistant")[-1].replace("<|im_end|>", "").strip()
    print("\n--- MODEL RESPONSE ---\n")
    print(clean_resp)
else:
    print("Run latest file not found. Please run training in Cell 5 first.")

## 7. Package and Download Trained LoRA Weights
Zips the trained LoRA adapter to `/kaggle/working/codeagent_7b_lora_weights.zip` for instant download.

In [ ]:
import shutil
output_zip = "/kaggle/working/codeagent_7b_lora_weights"
source_dir = ROOT / "training/models"

if source_dir.exists():
    shutil.make_archive(output_zip, "zip", source_dir)
    print(f"✓ Weights successfully zipped to: {output_zip}.zip")
    print("👉 You can now download this file directly from the Kaggle Output panel on the right side!")
else:
    print(f"Directory {source_dir} not found. Run training first.")